In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
# 改了 src/ 里的 .py，notebook 不用重启内核就能用上新代码。
# 魔法命令（% 开头）那一行后面不能写 # 注释，会报 UsageError，所以说明写在这里。
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_mysql import database as mysql
from signal_insight import config, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT  # 配色常量

viz.apply_theme()  # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)  # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY  # 分析日期（口径常量）
NB = "D03_空间维表"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"  # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
# ── 代码格 2：【跟丢了才运行】从检查点库恢复标准表 ──
from signal_mysql import database as mysql
# D2 没做完的同学：把下面一行的 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dq_check_result"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
# ── 代码格 3 · 需求卡 3.1：7 张静态表各做一张画像 ──
from signal_mysql import database as mysql
# 新拿到的 7 张静态表，先各做一张画像：行数、列数、哪些字段有缺失
static = ["ods_cell", "ods_poi_purpose", "ods_poi_transport", "ods_aoi_v1", "ods_aoi_v2", "ods_aoi_json", "ods_district_boundary"]
rows = []
for t in static:
    p = quality.profile(mysql.read_frame(f"SELECT * FROM {t}", con))
    gaps = p[p["n_missing"] > 0]
    rows.append({"表": t, "列数": len(p), "有缺失的字段": "、".join(f"{f}（{n}）" for f, n in zip(gaps["field"], gaps["n_missing"])) or "无"})
pd.DataFrame(rows)

In [ ]:
# ── 代码格 4 · 需求卡 3.1：基站表初识（扇区和站点）──
from signal_mysql import database as mysql
# 这一格在做什么：先认识基站表。信令里只有扇区编号（lac_cell），没有经纬度——
# **位置全靠这张表来补**，所以它的质量直接决定后面所有“在哪里”的结论。
cell = mysql.read_frame("SELECT * FROM ods_cell", con)
print(f"行数 {len(cell):,}；lac_cell 唯一值 {cell['lac_cell'].nunique():,}；坐标缺失 {cell[['lng', 'lat']].isna().any(axis=1).sum()} 行")
cell.describe().round(3)

In [ ]:
# ── 代码格 5 · 需求卡 3.1：把「扇区」聚合成「站点」──
from signal_mysql import database as mysql
# 这一格在做什么：把「扇区」聚合成「站点」。
#
# 一座基站铁塔上通常挂 3 个扇区天线，各朝一个方向覆盖 120°。它们的**经纬度完全相同**，
# 但编号（lac_cell）不同。所以：
# 扇区 = 表里的一行，是信令能定位到的最小粒度；
# 站点 = 坐标相同的扇区合成的一个物理位置。
# 讲台提示：不先把这件事说清楚，后面所有“有多少个基站”“基站有多密”的数都是错的。
cell = cell.dropna(subset=["lng", "lat"]).copy()                        # 没坐标的扇区定位不了，先去掉
sites = cell[["lng", "lat"]].drop_duplicates().reset_index(drop=True)   # 不重复的坐标 = 一个站点
sites["site_id"] = "S" + (sites.index + 1).astype(str).str.zfill(5)     # 编号补零到 5 位：S00001
cell = cell.merge(sites, on=["lng", "lat"], how="left")                 # 把站点号贴回每个扇区
per_site = cell.groupby("site_id").size()                               # 每个站点挂几个扇区
sector_dist = per_site.clip(upper=6).value_counts().sort_index().rename_axis("n_sectors").reset_index(name="n_sites")
sector_dist["sites_share"] = sector_dist["n_sites"] / sector_dist["n_sites"].sum()
mysql.write_frame(sector_dist, "ads_sectors_per_site", con, if_exists="replace", index=False)
print(f"扇区 {len(cell):,} 个，站点 {len(sites):,} 个，平均每个站点 {len(cell) / len(sites):.1f} 个扇区")

In [ ]:
# ── 代码格 6：看看每个站点挂几个扇区（中间表）──
sector_dist

In [ ]:
# ── 代码格 7 · 需求卡 3.1：画 D3_01「站点扇区数」──
from signal_mysql import database as mysql
sd = mysql.read_frame("SELECT * FROM ads_sectors_per_site", con)
mode = sd.loc[sd["n_sites"].idxmax()]
fig, ax = plt.subplots(figsize=(10, 5.6))
colors = [MAIN if k == mode["n_sectors"] else GRAY for k in sd["n_sectors"]]
bars = ax.bar([f"{k}+" if k == 6 else str(k) for k in sd["n_sectors"]], sd["sites_share"], color=colors, width=0.62)
ax.bar_label(bars, labels=[f"{p:.0%}" for p in sd["sites_share"]], padding=4, color=INK)
ax.set_ylim(0, sd["sites_share"].max() * 1.18)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("一个站点上的扇区数", color=MUTED)
title = f"{mode['sites_share']:.0%} 的站点挂着 {int(mode['n_sectors'])} 个扇区，同一站点的扇区坐标完全相同"
viz.finish(fig, title,
           f"按挂载扇区数统计的站点占比；站点 = 坐标相同的扇区；口径：ods_cell 中有坐标的 {len(cell):,} 个扇区",
           SOURCE, "D3_01_站点扇区数.png")

In [ ]:
# ── 代码格 8：对账 —— 扇区数、站点数 ──
assert len(cell) == 49_948 and len(sites) == 14_106
print("✅ 对账通过：49,948 个有坐标的扇区，14,106 个站点")

In [ ]:
# ── 代码格 9：把「基站坐标缺失」记进稽核表 ──
quality.dq_record(con, "DQ-10", "完整性", "基站坐标缺失", "ods_cell.lng / lat", "6 行", "通过",
          evidence=f"共 {len(cell) + 6:,} 行，缺坐标的占比不到 0.02%", decision="删除这 6 行")

In [ ]:
# ── 代码格 10 · 需求卡 3.2：检查信令能不能关联上基站表（关联率）──
from signal_mysql import database as mysql
# 这一格在做什么：把信令和基站表关联起来，并检查**关联率**。
# 两张表 join 之后一定要回头看“有多少行没匹配上”。关联率低而不自知，是企业里最常见的
# 数据事故之一——结果看着正常，其实悄悄少了一大块。
dwd = mysql.read_frame("SELECT * FROM dwd_signal", con, parse_dates=["ts"])   # 时间是文本存的，读回来要重新解析
assert dwd["ts"].notna().all(), "时间解析出了空值：检查 D2 写库前是否统一了时间格式"
# how="left"：保留全部信令。用 how="inner" 的话关联不上的行会直接消失，就查不出关联率了。
joined = dwd.merge(cell[["lac_cell", "site_id", "lng", "lat"]], on="lac_cell", how="left")
rec_rate = joined["lng"].notna().mean()
cells_used = dwd["lac_cell"].nunique()
miss = joined.loc[joined["lng"].isna(), "lac_cell"].drop_duplicates()
cells_missing = len(miss)
no_id = (~miss.isin(mysql.read_frame("SELECT lac_cell FROM ods_cell", con)["lac_cell"])).sum()   # 基站表里连编号都没有
print(f"按记录：{rec_rate:.4%} 能关联上坐标（对不上 {joined['lng'].isna().sum()} 条）")
print(f"按扇区：信令用到 {cells_used:,} 个扇区，{cells_missing} 个关联不到坐标：{no_id} 个基站表里没有这个编号，"
      f"{cells_missing - no_id} 个有编号但缺坐标")

In [ ]:
# ── 代码格 11：对账 —— 关联率 ──
assert rec_rate > 0.9999 and cells_missing == 8
print("✅ 对账通过：记录关联率超过 99.99%，8 个扇区关联不到坐标")

In [ ]:
# ── 代码格 12：把「关联率」记进稽核表 ──
quality.dq_record(con, "DQ-11", "一致性", "信令与基站表的关联率", "dwd_signal.lac_cell ↔ ods_cell.lac_cell",
          f"按记录 {rec_rate:.3%}", "通过",
          evidence=f"信令用到 {cells_used:,} 个扇区，{cells_missing} 个关联不到坐标（{no_id} 个基站表里没有编号，其余缺坐标）",
          decision="关联不上的记录在空间分析中剔除")

In [ ]:
# ── 代码格 13 · 需求卡 3.3：点落面 —— 每个站点在哪个区县 ──
from signal_mysql import database as mysql
from matplotlib.path import Path as MplPath

def rings_from_points(points, key):
    """从“一行一个顶点”的表还原多边形：{名称: [环1, 环2, ...]}，每个环是 N×2 的经纬度数组。
    points 由调用方读好传进来——函数里不读库（明天要搬进 geo.py）。"""
    pts = points if "ring" in points.columns else points.assign(ring=0)
    return {name: [r.sort_values("seq")[["lng", "lat"]].to_numpy(dtype=float) for _, r in g.groupby("ring")]
            for name, g in pts.groupby(key, sort=False)}

def point_in_polygons(lng, lat, polys):
    """点落面：返回每个点所在多边形的名称，不在任何多边形里的返回 None。"""
    xy = np.column_stack([np.asarray(lng, dtype=float), np.asarray(lat, dtype=float)])
    out = np.full(len(xy), None, dtype=object)
    for name, rings in polys.items():
        inside = np.zeros(len(xy), dtype=bool)
        for ring in rings:
            # ^= 是异或：被奇数个环包住 = 在里面，被偶数个环包住 = 在洞里。
            # 这样一个区县有飞地（多个外环）或有洞（内环）都能正确处理，不用分情况写。
            inside ^= MplPath(ring).contains_points(xy)
        out[inside & pd.isna(out)] = name
    return out

# 读库留在调用方：先把顶点表读出来，再交给函数还原
districts = rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
# 判断每个站点落在哪个区县。基站表覆盖的范围比沈阳大，落不进任何区县的统一标成「市外」——
# **不要丢掉**：下一格要用它来判断样本的地理代表性。
sites["district"] = point_in_polygons(sites["lng"], sites["lat"], districts)
sites["district"] = sites["district"].fillna("市外")
sites["district"].value_counts()

In [ ]:
# ── 代码格 14 · 需求卡 3.3：同一个「市外占比」，两个分母 ──
# 同一个「市外占比」，换个分母数字完全不同：
# 按**站点数**算 —— 基站表覆盖范围广，市外站点看着不少；
# 按**信令条数**算 —— 实际发生在市外的信令很少。
# 讲台提示：这是本课程「口径决定数字」的第一个例子，D8 会用六种算法把它推到极致。
# 两个数都要报，并且必须写清分母是谁。
out_share = (sites["district"] == "市外").mean()          # 分母：站点数
rec_district = joined.merge(sites[["site_id", "district"]], on="site_id", how="left")
rec_out = (rec_district["district"] == "市外").mean()     # 分母：信令条数
print(f"站点：{out_share:.1%} 在市外；信令记录：{rec_out:.1%} 发生在市外站点")

In [ ]:
# ── 代码格 15 · 需求卡 3.3：画 D3_02「站点与区县」──
def scale_bar(ax, km):
    """比例尺：按纬度把公里换成经度差。"""
    (x0_, x1_), (y0_, y1_) = ax.get_xlim(), ax.get_ylim()
    deg = km / (111.32 * np.cos(np.radians(41.8)))
    xs, ys = x0_ + 0.04 * (x1_ - x0_), y0_ + 0.05 * (y1_ - y0_)
    ax.plot([xs, xs + deg], [ys, ys], color=INK, lw=2, solid_capstyle="butt")
    ax.text(xs + deg / 2, ys + 0.015 * (y1_ - y0_), f"{km} 公里", ha="center", va="bottom", fontsize=8.5, color=INK)

def north_arrow(ax):
    ax.annotate("北", xy=(0.95, 0.93), xytext=(0.95, 0.83), xycoords="axes fraction", ha="center", va="center",
                fontsize=9, color=INK, arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2))

def draw_districts(ax, polys, values=None, cmap=None, norm=None, edge="#9AA3AE", lw=0.6):
    """区县边界；给了 values 就按数值填色（分级设色地图）。多边形由调用方传进来，函数里不读库。
    返回所有顶点的范围 (经度最小, 经度最大, 纬度最小, 纬度最大)，用来设视窗。"""
    for name, rings in polys.items():
        fc = cmap(norm(values[name])) if values is not None and name in values else "none"
        for ring in rings:
            ax.fill(ring[:, 0], ring[:, 1], facecolor=fc, edgecolor=edge, lw=lw)
    allpts = np.vstack([r for rings in polys.values() for r in rings])
    return allpts[:, 0].min(), allpts[:, 0].max(), allpts[:, 1].min(), allpts[:, 1].max()

lat0 = 41.8   # 沈阳大约的纬度，用来校正地图长宽比
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.5, 1])
bx0, bx1, by0, by1 = draw_districts(a1, districts, edge="#6B7280")
x0, x1 = bx0 - 0.15, bx1 + 0.15
y0, y1 = by0 - 0.12, by1 + 0.12
view = sites[sites["lng"].between(x0, x1) & sites["lat"].between(y0, y1)]
hidden = len(sites) - len(view)
inn, out = view[view["district"] != "市外"], view[view["district"] == "市外"]
a1.scatter(inn["lng"], inn["lat"], s=0.6, color=GRAY_DARK, lw=0, zorder=2)
a1.scatter(out["lng"], out["lat"], s=1.2, color=MAIN, lw=0, zorder=3)
a1.set_xlim(x0, x1); a1.set_ylim(y0, y1)
a1.set_aspect(1 / np.cos(np.radians(41.8)))   # 沈阳纬度约 41.8°：经度 1 度只有纬度的 0.75 倍长
a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)
for sp_ in a1.spines.values():
    sp_.set_visible(False)
scale_bar(a1, 20); north_arrow(a1)
# 右：同一个“市外”，两个分母
for i, (lab, v, note) in enumerate([("站点数", out_share, f"{(sites['district'] == '市外').sum():,} / {len(sites):,} 个站点"),
                                    ("信令条数", rec_out, "按 10-03 的信令记录计算")]):
    a2.barh(i, v, color=MAIN if i == 0 else GRAY, height=0.5)
    a2.text(v + 0.012, i, f"{v:.1%}" if v < 0.05 else f"{v:.0%}", va="center", fontsize=12, fontweight="bold", color=INK)
    a2.text(0, i + 0.42, note, va="center", fontsize=8.8, color=MUTED)
a2.set_yticks([0, 1], ["站点数", "信令条数"], fontsize=10, color=INK)
a2.invert_yaxis(); a2.set_xlim(0, 0.5); a2.set_xticks([]); a2.grid(False); a2.spines["bottom"].set_visible(False)
a2.set_title("市外占比：按站点 vs 按信令", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"{len(sites):,} 个站点中 {out_share:.0%} 在沈阳市外，但只有 {rec_out:.1%} 的信令发生在那里"
viz.finish(fig, title,
           f"蓝点 = 市外站点，灰点 = 市内站点（坐标相同的扇区合并为一个站点）；视窗外另有 {hidden:,} 个站点；口径：dim_cell、dwd_signal（10-03）",
           SOURCE, "D3_02_站点与区县.png")

In [ ]:
# ── 代码格 16：对账 + 把「基站表的地理范围」记进稽核表 ──
assert (sites["district"] == "市外").sum() == 4_808 and 0.015 < rec_out < 0.02
print("✅ 对账通过：4,808 个站点在市外；市外站点上的信令不到 2%")
quality.dq_record(con, "DQ-12", "代表性", "基站表的地理范围", "ods_cell",
          f"{out_share:.0%} 的站点在沈阳市外", "通过",
          evidence=f"只有 {rec_out:.1%} 的信令发生在市外站点", decision="保留市外站点，区县标为“市外”")

In [ ]:
# ── 代码格 17 · 需求卡 3.4：三份商圈边界，先各自体检一遍 ──
from signal_mysql import database as mysql
# 这一格在做什么：库里有三份商圈边界（v1、v2、json 版），先各自体检一遍。
# 判断标准**不是“哪个新”**，而是“哪个经得起检验”：
#   ① 坐标在不在合理范围内（沈阳大约在东经 123 度、北纬 41.8 度）；
#   ② 面积合不合理；
#   ③ 地铁站这类**已知位置**的点，落不落在它该在的商圈里。
# 同名数据有多个版本是企业里的常态，这套检验方法比“问谁要最新的”可靠得多。
v1_pts = mysql.read_frame("SELECT * FROM ods_aoi_v1", con)
json_pts = mysql.read_frame("SELECT * FROM ods_aoi_json", con)
print("v1 里经度小于 100 的点：")
print(v1_pts[v1_pts["lng"] < 100])
print("\njson 版本的坐标范围（看单位）：")
print(json_pts[["x", "y"]].describe().loc[["min", "max"]])

In [ ]:
# ── 代码格 18 · 需求卡 3.4：算面积 + 用地铁站当「裁判」──
from signal_mysql import database as mysql

def ring_area_km2(ring):
    """多边形面积（平方公里）：鞋带公式，按所在纬度把经纬度换算成公里。"""
    x, y = ring[:, 0], ring[:, 1]
    kx, ky = 111.32 * np.cos(np.radians(y.mean())), 110.57        # 每度经度、每度纬度约多少公里
    return abs(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)) / 2 * kx * ky

aoi_v1 = rings_from_points(v1_pts, "name")           # 上一格已经读出来了
# v1 里有个点的经度写成了 23.45（开头的 1 掉了），直接算面积会得到一个横跨半个中国的多边形。
# 先把这类明显写错的点剔掉再比，才算公平——否则结论会变成“v1 面积离谱”，而真正的毛病
# 是顶点顺序错乱（连出来是交叉的折线）。
aoi_v1 = {k: [r[r[:, 0] > 100] for r in rings] for k, rings in aoi_v1.items()}
aoi_v2 = rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
# 地铁站当「裁判」：它们的位置是公认的，太原街商圈理应包含太原街地铁站。
metro = mysql.read_frame("SELECT * FROM ods_poi_transport WHERE type = '地铁'", con)
metro["v1"] = point_in_polygons(metro["lng"], metro["lat"], aoi_v1)
metro["v2"] = point_in_polygons(metro["lng"], metro["lat"], aoi_v2)

In [ ]:
# ── 代码格 19 · 需求卡 3.4：两个版本逐项对比，写回 ads_aoi_check ──
aoi_check = pd.DataFrame([{
    "aoi": name,
    "v1_area_km2": round(sum(ring_area_km2(r) for r in aoi_v1[name]), 2),
    "v2_area_km2": round(sum(ring_area_km2(r) for r in aoi_v2[name]), 2),
    "v1_metro_stations": "、".join(metro.loc[metro["v1"] == name, "name"].str.replace("(地铁站)", "", regex=False)) or "无",
    "v2_metro_stations": "、".join(metro.loc[metro["v2"] == name, "name"].str.replace("(地铁站)", "", regex=False)) or "无",
} for name in aoi_v2])
mysql.write_frame(aoi_check, "ads_aoi_check", con, if_exists="replace", index=False)
aoi_check

In [ ]:
# ── 代码格 19 续 · 需求卡 3.4：画 D3_03「商圈边界版本对比」──
show = ["太原街商圈", "中街商圈", "沈阳北站"]
fig, axes = plt.subplots(1, 3, figsize=(10, 5.6))
for ax, name in zip(axes, show):
    ring2 = aoi_v2[name][0]
    ax.fill(ring2[:, 0], ring2[:, 1], color=MAIN_LIGHT, lw=0)
    ax.plot(ring2[:, 0], ring2[:, 1], color=MAIN, lw=1.6)
    for r1 in aoi_v1[name]:
        ax.plot(r1[:, 0], r1[:, 1], color=RED, lw=1.4, ls=(0, (3, 2)))
    pad = 0.006
    x0, x1 = ring2[:, 0].min() - pad, ring2[:, 0].max() + pad
    y0, y1 = ring2[:, 1].min() - pad, ring2[:, 1].max() + pad
    near = metro[metro["lng"].between(x0, x1) & metro["lat"].between(y0, y1)]
    ax.scatter(near["lng"], near["lat"], s=22, color=INK, zorder=3)
    for _, m in near.iterrows():
        label = m["name"].replace("(地铁站)", "")
        label = label if label.endswith("站") else label + "站"
        ax.annotate(label, (m["lng"], m["lat"]), xytext=(4, 4),
                    textcoords="offset points", fontsize=8.5, color=INK)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect(1 / np.cos(np.radians(41.8)))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    a = aoi_check.set_index("aoi").loc[name]
    ax.set_title(f"{name}\nv1 {a['v1_area_km2']} km² ｜ v2 {a['v2_area_km2']} km²", loc="left", fontsize=10.5, color=INK)
tyj = aoi_check.set_index("aoi").loc["太原街商圈"]
title = f"v1 的太原街商圈只有 {tyj['v1_area_km2']} km²，连太原街地铁站都没圈进去"
viz.finish(fig, title,
           "蓝色实线 = v2 边界；红色虚线 = v1 边界（已剔除写错的点；顶点顺序错乱，连出来是交叉的折线）；黑点 = 地铁站",
           SOURCE, "D3_03_商圈边界版本对比.png")

In [ ]:
# ── 代码格 20：对账 ──
a = aoi_check.set_index("aoi")
assert a.loc["太原街商圈", "v1_area_km2"] < 0.1 and a.loc["太原街商圈", "v1_metro_stations"] == "无"
assert a.loc["太原街商圈", "v2_metro_stations"] == "沈阳站、太原街"
assert a.loc["中街商圈", "v2_metro_stations"] == "中街、东中街" and a.loc["沈阳北站", "v2_metro_stations"] == "沈阳北站"
print("✅ 对账通过：v1 太原街面积异常且漏掉地铁站；v2 三个商圈都圈进了对应的地铁站")

In [ ]:
# ── 代码格 21 · 需求卡 3.4：把两个版本的问题记进稽核表 ──
quality.dq_record(con, "DQ-13", "准确性", "商圈边界 v1", "ods_aoi_v1",
          f"太原街商圈面积 {tyj['v1_area_km2']} km²，未包含太原街地铁站", "不通过",
          evidence="沈阳北站有一个点经度写成 23.45；多边形顶点顺序错乱", decision="弃用 v1，采用 v2")
quality.dq_record(con, "DQ-14", "规范性", "商圈边界 json 版本的坐标单位", "ods_aoi_json",
          f"坐标在 {json_pts['x'].min():,.0f}～{json_pts['x'].max():,.0f} 之间，是米制投影坐标", "警告",
          evidence="数值远超经纬度范围，需要换算坐标系才能使用", decision="不使用；需要时向数据提供方确认坐标系")

In [ ]:
# ── 代码格 22 · 需求卡 3.5：写回维度表 dim_cell ──
from signal_mysql import database as mysql
# 这一格在做什么：把今天的结果合成维度表 dim_cell——**一个扇区一行**，
# 带上站点号、坐标、所在区县、所在商圈。后面所有“在哪里”的分析都 join 这张表。
sites["aoi"] = point_in_polygons(sites["lng"], sites["lat"], aoi_v2)           # 用甄别后选定的 v2 边界
dim_cell = cell.merge(sites[["site_id", "district", "aoi"]], on="site_id", how="left")
dim_cell = dim_cell[["lac_cell", "site_id", "lng", "lat", "district", "aoi"]]   # 固定列顺序，下游看着稳定
# ⚠ 维度表的主键必须唯一。这里要是重了，后面每 join 一次就会把信令行数放大一倍，
# 而且不报错——所有人数、条数会在毫无征兆的情况下虚高。断言放在写库前，当场挡住。
assert dim_cell["lac_cell"].is_unique
mysql.write_frame(dim_cell, "dim_cell", con, if_exists="replace", index=False)
print(mysql.read_frame("SELECT COUNT(*) AS n FROM dim_cell", con))
dim_cell.groupby("aoi")["site_id"].nunique().rename("n_sites")

In [ ]:
# ── 代码格 23：对账 ──
assert len(dim_cell) == 49_948 and dim_cell["aoi"].notna().sum() > 0
print("✅ 对账通过：dim_cell 49,948 行，主键唯一")

In [ ]:
# ── 代码格 24 · 需求卡 3.7（选讲）：坐标系一致性检验 ──
def haversine(lat1, lng1, lat2, lng2):
    """两点间的球面距离（米）。参数可以是数字，也可以是等长的数组。"""
    lat1, lng1, lat2, lng2 = (np.radians(np.array(v, dtype=float)) for v in (lat1, lng1, lat2, lng2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

def nearest_site_m(lng, lat):
    """每个点到最近站点的距离（米）：点数 × 站点数的距离矩阵，取每行最小值。"""
    d = haversine(np.array(lat)[:, None], np.array(lng)[:, None], sites["lat"].values[None, :], sites["lng"].values[None, :])
    return d.min(axis=1)

as_is = nearest_site_m(metro["lng"], metro["lat"])
shifted = nearest_site_m(metro["lng"] - 0.0061, metro["lat"] - 0.0024)      # 模拟“坐标系差一档”：沈阳一带约偏 580 米
print(f"原样：地铁站到最近站点中位 {np.median(as_is):.0f} 米，{(as_is < 100).mean():.0%} 在 100 米内")
print(f"整体平移约 580 米后：中位 {np.median(shifted):.0f} 米，{(shifted < 100).mean():.0%} 在 100 米内")

In [ ]:
# ── 代码格 25 · 需求卡 3.8（选讲）：主键的第二个证据——“同组”的人在不在一起 ──
loc = joined.dropna(subset=["lng"]).assign(hour=lambda x: x["ts"].dt.hour, imsi15=lambda x: x["imsi"].str[:15])
multi = loc.groupby("imsi15")["imsi"].nunique()
grp = loc[loc["imsi15"].isin(multi[multi > 1].index)]
hourly_pos = grp.groupby(["imsi15", "imsi", "hour"])[["lng", "lat"]].median().reset_index()
pairs = hourly_pos.merge(hourly_pos, on=["imsi15", "hour"])
pairs = pairs[pairs["imsi_x"] < pairs["imsi_y"]]
pairs["dist"] = haversine(pairs["lat_x"], pairs["lng_x"], pairs["lat_y"], pairs["lng_y"])
typical = pairs.groupby(["imsi15", "imsi_x", "imsi_y"])["dist"].median()
print(f"共 {len(typical):,} 对“同组”IMSI；同一小时相距的中位数 {typical.median() / 1000:.1f} 公里；"
      f"{(typical > 2000).mean():.0%} 的对相距超过 2 公里")

In [ ]:
# ── 代码格 26 · 需求卡 3.9（选讲）：基站有多密——站点间距 ──
from signal_mysql import database as mysql
site = (mysql.read_frame("SELECT site_id, lng, lat, district FROM dim_cell", con)
        .groupby("site_id").agg(lng=("lng", "first"), lat=("lat", "first"), district=("district", "first")))
lng0, lat0 = 123.4, 41.8                                       # 沈阳市区大约在东经 123.4 度、北纬 41.8 度
kx, ky = 111.320 * np.cos(np.radians(lat0)), 110.574           # 城市尺度下，经纬度可以近似换成平面公里（1 度约多少公里）
xy = np.column_stack([(site["lng"] - lng0) * kx, (site["lat"] - lat0) * ky]).astype(np.float32)
nn = np.empty(len(xy), dtype=np.float32)
# 分块算：14,106 个站点两两算距离是 2.0×10⁸ 个数，一次性展开要 0.8 GB 内存，白白占着还容易爆。
# 每次只取 500 个站点对全部站点（500 × 14,106 ≈ 700 万个数，约 28 MB），循环 29 轮。
# 讲台提示：这是本课程第一次出现“算不动怎么办”。D5 还会讲另外两招：先筛后算、用 SQL 聚合。
for s0 in range(0, len(xy), 500):
    a = xy[s0:s0 + 500]
    d2 = (a[:, :1] - xy[None, :, 0]) ** 2 + (a[:, 1:] - xy[None, :, 1]) ** 2
    d2[np.arange(len(a)), np.arange(s0, s0 + len(a))] = np.inf    # 排除自己
    nn[s0:s0 + 500] = np.sqrt(d2.min(axis=1))
site["nn_m"] = nn * 1000
area = {}                                                      # 各区县面积（平方公里），用鞋带公式
for name, rings in districts.items():
    area[name] = sum(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                     for x, y in (((r[:, 0] - lng0) * kx, (r[:, 1] - lat0) * ky) for r in rings))
by = (site[site["district"] != "市外"].groupby("district")
        .agg(n_sites=("nn_m", "size"), spacing_m_p50=("nn_m", "median")).sort_values("spacing_m_p50"))
by["sites_per_km2"] = by["n_sites"] / pd.Series(area)
by.round(1)

In [ ]:
# ── 代码格 27 · 需求卡 3.9：画 D3_04「基站间距」──
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(by)), by["spacing_m_p50"], color=[MAIN if k == "和平区" else GRAY for k in by.index], height=0.66)
ax.set_yticks(range(len(by)), by.index)
ax.invert_yaxis()
for i, (k, r_) in enumerate(by.iterrows()):
    ax.text(r_["spacing_m_p50"] + 18, i, f"{r_['spacing_m_p50']:,.0f} 米（{r_['sites_per_km2']:.1f} 个/平方公里）", va="center",
            fontsize=9.5, color=MAIN if k == "和平区" else INK)
ax.axvline(500, color=INK, lw=0.9, ls="--")
ax.set_ylim(len(by) - 0.4, -1.4)
ax.text(515, -1.0, "D6 停留识别的半径 500 米", fontsize=9.5, color=INK, va="center")
ax.set_xlim(0, by["spacing_m_p50"].max() * 1.5)
ax.set_xticks([]); ax.grid(False)
hp, far = by.loc["和平区", "spacing_m_p50"], by["spacing_m_p50"].idxmax()
title = f"和平区站点间距中位 {hp:.0f} 米，{far} {by.loc[far, 'spacing_m_p50']:,.0f} 米，相差 {by.loc[far, 'spacing_m_p50'] / hp:.0f} 倍"
viz.finish(fig, title,
           "每个站点到最近站点的直线距离，按站点所在区县取中位数，括号内为站点密度；\n"
           "市中心 500 米内有十几个站点，郊县可能一个都没有：同一套参数，在城区和郊县的含义不同；口径：dim_cell",
           SOURCE, "D3_04_基站间距.png")

In [ ]:
# ── 代码格 28：收尾 —— 关掉连接 ──
con.close()